# GPU Hardware and Multi-GPU Interconnects

> In previous sections discussing DDP, ZeRO, and FSDP, we treated the GPU as an abstract "compute unit" — give it data, and it returns gradients. But in reality, a GPU has a clear internal memory hierarchy, and multiple GPUs are connected in several ways with vastly different bandwidths. Abstracting away these hardware details lets us explain algorithms clearly; once we move into real training and inference, the hardware topology often determines whether a scheme is feasible at all.
>
> This section starts inside a single GPU: why the gap between HBM and SRAM in capacity and bandwidth makes Flash Attention a necessity. We then expand to multi-GPU interconnects, comparing the bandwidth magnitudes of PCIe, NVLink, NVSwitch, and InfiniBand to understand the topology of a DGX H100. Finally, we discuss topology-aware scheduling, and why Expert Parallelism in MoE is especially sensitive to network topology.


A GPU is not a homogeneous block of "big memory + fast compute." Inside an H100, the SRAM close to the compute units has only about 20 MB, but bandwidth of tens of TB/s; the HBM farther from the compute units has 80 GB, but bandwidth of only about 3 TB/s. The two differ by more than 30x. This gap determines which computations can run fast — keeping data in SRAM as much as possible and avoiding repeated reads and writes to HBM is the core idea behind high-performance CUDA kernel design.

At the multi-GPU level, PCIe is a general-purpose bus with limited bandwidth, NVLink is a dedicated high-speed channel between GPUs, and InfiniBand is a cross-node network protocol. For "transferring 1 GB of data between two GPUs," going over NVLink versus PCIe can differ in speed by 6 to 10 times. In distributed training, communication time often accounts for a significant fraction of total time, so understanding multi-GPU topology is not the exclusive domain of hardware engineers — it is a foundation that anyone who wants to run large models must master.

This section proceeds in the order "single-GPU memory hierarchy -> GPU internal structure -> multi-GPU interconnects -> typical topology -> topology-aware scheduling," and finally demonstrates how to inspect this information on a real machine using tools like `nvidia-smi`.


## 1. Single-GPU Memory Hierarchy: HBM and SRAM

A GPU internally has two main storage tiers. HBM (High Bandwidth Memory) is the device memory — large capacity but relatively slow; SRAM (Static Random Access Memory) is the on-chip cache — small capacity but extremely fast. In NVIDIA architectures, SRAM mainly refers to the L1 cache and shared memory, collectively called "on-chip memory."

The table below lists the memory configurations of several mainstream modern datacenter GPUs:


In [ ]:
# === Memory configurations of mainstream datacenter GPUs ===
# Source: NVIDIA official specification sheets

gpus = [
    # (model, HBM capacity GB, HBM bandwidth TB/s, SRAM capacity MB, release year)
    ("A100 40GB",  40, 1.55,  20, 2020),
    ("A100 80GB",  80, 2.00,  20, 2021),
    ("H100 SXM",   80, 3.35,  20, 2022),
    ("H200 SXM",  141, 4.80,  20, 2024),
    ("B200",      192, 8.00,  40, 2024),
]

print(f"{'Model':<14} {'HBM(GB)':>9} {'HBM BW(TB/s)':>15} {'SRAM(MB)':>10} {'Year':>6}")
print("-" * 60)
for name, hbm_gb, hbm_bw, sram_mb, year in gpus:
    print(f"{name:<14} {hbm_gb:>9} {hbm_bw:>15.2f} {sram_mb:>10} {year:>6}")

print()
print("Key observation: from A100 to B200, HBM capacity grew 5x and bandwidth grew about 5x, but SRAM grew slowly.")
print("Larger HBM can hold larger KV caches and activations; higher bandwidth speeds up reads/writes of large tensors.")


### The Bandwidth Gap Between HBM and SRAM

HBM bandwidth looks high (H100 is 3.35 TB/s), but SRAM is even higher. The combined usable bandwidth of H100's shared memory plus L1 cache is estimated at the 30 TB/s level, almost 10x faster than HBM. This gap directly affects how attention is implemented.


In [ ]:
# === HBM vs SRAM bandwidth comparison (H100 example) ===
hbm_bw_tb = 3.35       # H100 HBM bandwidth, in TB/s
sram_bw_tb = 30.0      # H100 SRAM estimated bandwidth, in TB/s (approximate)

ratio = sram_bw_tb / hbm_bw_tb
print(f"H100 HBM bandwidth:  {hbm_bw_tb:.2f} TB/s")
print(f"H100 SRAM bandwidth: {sram_bw_tb:.2f} TB/s (estimated)")
print(f"Bandwidth gap: SRAM is {ratio:.1f}x of HBM")
print()
print("If an attention kernel needs to repeatedly read/write an N x N attention matrix,")
print("going through HBM each time is much slower than tiling the matrix and keeping it in SRAM.")
print("This is exactly the starting point of Flash Attention: tiled computation, keeping intermediate results in SRAM.")


### Why Flash Attention Is Necessary

The standard attention formula is $\text{softmax}(QK^T / \sqrt{d}) V$. A naive implementation first writes the $N \times N$ attention score matrix to HBM, then reads it back to compute softmax. As the sequence length $N$ grows, the memory footprint and read/write traffic of this $N^2$ matrix both explode.

The core idea of Flash Attention is to tile Q, K, and V into blocks where each block's intermediate results are small enough to fit in SRAM; softmax is computed block by block using an online algorithm (accumulate and normalize on the fly), without ever materializing the full $N^2$ matrix to HBM. This reduces attention's HBM read/writes from $O(N^2)$ to $O(N)$, and the activation memory also drops to $O(N)$. This is precisely an algorithm design forced by the hardware characteristic that SRAM is limited but extremely fast.


## 2. GPU Internal Structure: SM, Tensor Core, Warp

HBM and SRAM are only the storage side. On the compute side, a GPU is composed of several SMs (Streaming Multiprocessors). Each SM contains a number of CUDA cores, dedicated Tensor Cores, a register file, and shared memory. The H100 SXM has 132 SMs, each with 256 KB of shared memory and 256 KB of registers.

The scheduling unit is the warp: a warp is 32 threads, and an SM issues and manages several warps at once. When writing CUDA, programmers usually do not deal with warps directly, but for high-performance kernels data must be organized with 32-way alignment to avoid warp branch divergence.


In [ ]:
# === Compute resources of H100 SXM ===
sm_count = 132                  # Number of SMs
shared_mem_per_sm_kb = 256      # Shared memory per SM
register_per_sm_kb = 256        # Register file per SM

print(f"Key parameters of H100 SXM:")
print(f"  Number of SMs:           {sm_count}")
print(f"  Shared memory per SM:    {shared_mem_per_sm_kb} KB")
print(f"  Registers per SM:        {register_per_sm_kb} KB")
print(f"  Total shared memory:     {sm_count * shared_mem_per_sm_kb / 1024:.1f} MB ~= total SRAM")
print()
print(f"Warp size is fixed at 32 threads (same across all NVIDIA architectures).")
print(f"When writing kernels, accessing memory with 32-way alignment lets all threads in a warp execute together.")


### Tensor Cores and FP8

Tensor Cores are dedicated matrix-multiply units introduced by NVIDIA starting with the Volta architecture (V100, 2017). A single Tensor Core instruction (MMA, Matrix Multiply-Accumulate) performs a small matrix multiply-add at the hardware level: $D = A \times B + C$, where $A, B, C, D$ are $16\times16$ or smaller fragments.

Tensor Cores on different architectures support different data types:

| Architecture | Representative GPU | Key data types |
|:---|:---|:---|
| Volta | V100 | FP16 |
| Ampere | A100 | FP16, BF16, TF32, INT8 |
| Hopper | H100 | Above + FP8 (E4M3, E5M2) |
| Blackwell | B200 | Above + FP4, FP6 |

Hopper's introduction of FP8 Tensor Cores is a key milestone for LLM training and inference. FP8 uses 8 bits to represent a floating-point number — compared to FP16, both memory footprint and bandwidth are halved; compared to BF16, it preserves a similar numeric range (E4M3 has 4 exponent bits + 3 mantissa bits; E5M2 has 5 exponent bits + 2 mantissa bits). Within a suitable data range, FP8 training can achieve accuracy close to BF16 while running much faster.


In [ ]:
# === Comparison of the two FP8 formats ===
# E4M3: 1 sign + 4 exponent + 3 mantissa -> higher precision, smaller range, suitable for forward pass
# E5M2: 1 sign + 5 exponent + 2 mantissa -> lower precision, larger range, suitable for backward gradients

print(f"{'Format':<8} {'Sign':>6} {'Exp':>6} {'Mant':>6} {'Typical use':<22}")
print("-" * 55)
print(f"{'E4M3':<8} {'1':>6} {'4':>6} {'3':>6} {'Forward weights':<22}")
print(f"{'E5M2':<8} {'1':>6} {'5':>6} {'2':>6} {'Backward gradients':<22}")
print()
print("Benefits of FP8 over BF16:")
print("  - Memory: the same tensor needs only half the bytes (1 byte vs 2 bytes)")
print("  - Bandwidth: HBM read/write volume is halved, a clear win for bandwidth-bound ops")
print("  - Compute: H100 FP8 Tensor Core peak throughput is 2x that of BF16")
print()
print("Cost: narrow numeric range requires per-tensor or per-row scale factors to avoid overflow.")


## 3. Multi-GPU Interconnects: PCIe, NVLink, NVSwitch, InfiniBand

Single-GPU training is limited by memory and compute, so any moderately sized model must use multiple GPUs. The bandwidth for moving data (gradients, parameters, KV cache) between GPUs then becomes the bottleneck. The NVIDIA ecosystem has four mainstream connection types, in increasing order of bandwidth magnitude: PCIe, NVLink, NVSwitch, and InfiniBand (the latter two have different roles).


In [ ]:
# === Bandwidth magnitude comparison of multi-GPU interconnects ===
# Values from NVIDIA public specs (typical values, unidirectional bandwidth)

interconnects = [
    # (name, unidirectional bandwidth GB/s, scope, typical use)
    ("PCIe 4.0 x16",       32,  "intra-node",  "GPU-CPU, GPU-NVMe, low-end GPU-GPU"),
    ("PCIe 5.0 x16",       64,  "intra-node",  "H100 PCIe version to CPU interconnect"),
    ("NVLink 3.0",        300,  "intra-node",  "Direct link between A100s"),
    ("NVLink 4.0",        450,  "intra-node",  "Direct link between H100s"),
    ("NVLink 5.0",        900,  "intra-node",  "Direct link between B200s"),
    ("InfiniBand HDR",     50,  "cross-node",  "200 Gbps network conversion"),
    ("InfiniBand NDR",    100,  "cross-node",  "400 Gbps network conversion"),
    ("InfiniBand XDR",    200,  "cross-node",  "800 Gbps network conversion"),
]

print(f"{'Name':<20} {'BW(GB/s)':>12} {'Scope':>12}  Typical use")
print("-" * 85)
for name, bw, scope, use in interconnects:
    print(f"{name:<20} {bw:>12} {scope:>12}  {use}")

print()
print("Key observation: intra-node NVLink is 6-10x of PCIe; cross-node InfiniBand is 4-9x slower than intra-node NVLink.")
print("This means the communication speed difference between 'two GPUs in the same node' and 'two GPUs across nodes' is huge.")


### Why NVLink Is 6 to 10 Times Faster Than PCIe

PCIe is a general-purpose bus standard, originally designed for the CPU and various peripherals (network cards, graphics cards, disks). The protocol itself has significant overhead — each transfer goes through multiple layers such as the PCIe controller and root complex. NVLink is a protocol designed by NVIDIA specifically for point-to-point communication between GPUs, using differential signaling lines to directly connect the dies of two GPUs, bypassing the PCIe controller, and supporting cache coherence and direct memory access (Peer-to-Peer, P2P).

In physical implementation, each H100 SXM chip has 4 NVLink 4.0 chips, each providing 50 GB/s of unidirectional bandwidth, for a total of 200 GB/s unidirectional and 450 GB/s bidirectional (NVIDIA's convention is to count bidirectionally). B200 raises per-chip bandwidth to 100 GB/s, bringing total NVLink bandwidth to 900 GB/s bidirectional. By comparison, PCIe 5.0 x16 has only 64 GB/s unidirectional — the physical bandwidth ceiling differs by an order of magnitude.


In [ ]:
# === Bandwidth gap between NVLink and PCIe (H100 example) ===
nvlink4_bidir = 450     # NVLink 4.0 bidirectional bandwidth GB/s
pcie5_unidir = 64       # PCIe 5.0 x16 unidirectional bandwidth GB/s

# Data movement demo: transferring BF16 parameters of a 70B model between two GPUs
P = 70e9
param_bytes = 2 * P     # BF16 is 2 bytes per param

t_nvlink = param_bytes / (nvlink4_bidir * 1e9)
t_pcie   = param_bytes / (pcie5_unidir * 1e9)

print(f"Transferring BF16 params of a 70B model (~{param_bytes / 1e9:.0f} GB):")
print(f"  Over NVLink 4.0: {t_nvlink:.2f} s")
print(f"  Over PCIe 5.0:   {t_pcie:.2f} s")
print(f"  Speed gap: {t_pcie / t_nvlink:.1f}x")
print()
print("This is why high-bandwidth training almost always uses NVLink; PCIe is only used in low-cost scenarios or on the inference side.")


### NVSwitch: Intra-Node Full Interconnect

NVLink is a point-to-point protocol: one NVLink channel between two GPUs. When the number of GPUs in a node grows, running NVLink between every pair makes the topology very complex. The 8-GPU nodes of the H100 generation have 28 GPU pairs to interconnect ($C_8^2 = 28$); fully wiring NVLink for every pair would require a large number of chips.

NVSwitch is NVIDIA's solution: place one or more NVSwitch chips in the node, and each GPU connects to the switch via NVLink, with the switch handling forwarding. This way any two GPUs can run at full NVLink bandwidth without dedicated wiring per pair. The DGX H100 has 4 third-generation NVSwitches, forming a full all-to-all topology among the 8 H100s.

By analogy, NVLink is like a direct cable between two machines, while NVSwitch is like plugging them all into a switch.


### InfiniBand: The Cross-Node Network

With NVSwitch solving full interconnect within a node, what about cross-node? Standard Ethernet has latency that is too high and congestion control that is not predictable enough for large-scale distributed training. InfiniBand (IB) is a network protocol designed specifically for high-performance computing, characterized by low latency (microsecond level), high bandwidth (HDR 200 Gbps, NDR 400 Gbps, XDR 800 Gbps), and hardware-level reliable transport.

Each GPU node is typically equipped with 4 to 8 IB network cards (HCAs, Host Channel Adapters). The network cards connect to the CPU via PCIe 5.0, or connect directly to GPUs via NVLink Switch (this is the approach of NVIDIA's Magnum IO architecture, called "GPU Direct RDMA"). GPU Direct RDMA lets one GPU's memory write directly through the IB network card to a GPU's memory on another node, bypassing the CPU memory hop and further reducing latency.


In [ ]:
# === Cross-node transfer: sending a 70B model over IB NDR ===
# NDR single link 400 Gbps = 50 GB/s

ib_ndr_gbps = 400          # Gbps
ib_ndr_gbs = ib_ndr_gbps / 8   # Convert to GB/s

P = 70e9
param_bytes = 2 * P

# Typically 8 IB cards per node, aggregate bandwidth 8 x 50 = 400 GB/s
n_hca = 8
agg_bw = n_hca * ib_ndr_gbs

t_single = param_bytes / (ib_ndr_gbs * 1e9)
t_agg = param_bytes / (agg_bw * 1e9)

print(f"IB NDR single link: {ib_ndr_gbs:.0f} GB/s ({ib_ndr_gbps} Gbps)")
print(f"8 IB cards aggregate: {agg_bw:.0f} GB/s")
print()
print(f"Cross-node transfer of 70B BF16 params ({param_bytes / 1e9:.0f} GB):")
print(f"  Single link:    {t_single:.2f} s")
print(f"  8-link aggregate: {t_agg:.2f} s")
print()
print("Compared to 1.16 s over intra-node NVLink, cross-node is about 2-3x slower (even saturating 8 IB cards).")
print("This is why the choice of expert parallelism and tensor parallelism must carefully consider topology.")


## 4. Typical Topology: DGX H100

Putting the connection types above together, let us look at the complete topology of a DGX H100 node. The DGX H100 is NVIDIA's official high-density training node, with 8 H100 SXM GPUs per node — the "standard training unit" of this era.


### DGX H100 Intra-Node Topology

Key structure of a single DGX H100 node:

- 8 H100 SXM GPUs, each with 80 GB HBM
- 4 third-generation NVSwitch chips, forming an 8-GPU full interconnect
- Each H100 has 4 NVLink 4.0 chips, totaling 900 GB/s bidirectional bandwidth (actual configuration counted as 450 GB/s bidirectional)
- 2 Intel Sapphire Rapids CPUs, connecting GPUs and peripherals via PCIe 5.0
- 8 ConnectX-7 IB network cards (400 Gbps NDR), one per GPU
- 2 BlueField-3 DPUs, handling storage and network protocol offload

This design gives any two GPUs in the node 450 GB/s of bidirectional bandwidth; cross-node, via IB NDR single link at 50 GB/s, each node aggregates 8 links for 400 GB/s.


In [ ]:
# === DGX H100 intra-node vs cross-node bandwidth comparison ===
# Intra-node: 8-GPU full interconnect
intra_link = 450          # NVLink bidirectional GB/s between any two GPUs
n_pairs_intra = 8 * 7 // 2   # C(8,2) = 28 pairs
intra_agg = intra_link * n_pairs_intra

# Cross-node: 8 IB NDR cards per node
ib_link = 50              # Single link GB/s
n_hca = 8
inter_agg = ib_link * n_hca

print(f"DGX H100 intra-node full-interconnect total bandwidth: {intra_agg} GB/s")
print(f"  (28 GPU pairs x 450 GB/s = {intra_agg} GB/s)")
print()
print(f"DGX H100 cross-node aggregate bandwidth: {inter_agg} GB/s")
print(f"  (8 IB cards x 50 GB/s = {inter_agg} GB/s)")
print()
print(f"Intra-node bandwidth is {intra_agg / inter_agg:.1f}x of cross-node.")
print("This is the fundamental reason for 'topology-aware scheduling': keep communication-heavy operations within the node.")


### Cluster Network Hierarchy

Connecting multiple DGX nodes into a training cluster, the network has three layers:

- **Intra-node**: NVLink + NVSwitch, full interconnect, highest bandwidth
- **Intra-rack**: typically 2-4 DGX nodes per rack, interconnected via IB switches
- **Cross-rack**: multiple racks connected via spine switches, bandwidth further aggregated

A 10,000-GPU-scale training cluster typically has about 1,000-1,500 DGX nodes spread across dozens of racks. This multi-layer structure means the communication path between any two GPUs can be completely different — intra-node is NVLink, intra-rack is 1-2 IB hops, cross-rack is 3-4 IB hops. Both latency and bandwidth drop layer by layer.


In [ ]:
# === Cluster network hierarchy bandwidth estimation ===
layers = [
    # (layer, single-hop bandwidth GB/s, typical hop count)
    ("Intra-node (NVLink)",   450, 1),
    ("Intra-rack (IB NDR)",   50, 1),
    ("Cross-rack (spine)",    50, 2),
]

print(f"{'Layer':<26} {'Per-hop BW':>14} {'Typical total BW':>18}")
print("-" * 65)
for name, bw, hops in layers:
    eff = bw / hops
    print(f"{name:<26} {bw:>10} GB/s {eff:>14.1f} GB/s")

print()
print("Bandwidth drops layer by layer, so the scheduler should keep communication-heavy tasks within the node whenever possible.")


## 5. Topology-Aware Scheduling

Having understood the bandwidth gaps, we can explain why distributed training schedulers must be "topology-aware." A naive idea is to randomly assign N GPUs to a training job. But this may place two ranks that need to communicate frequently across nodes, causing every training step to wait on cross-node IB.

The right approach is to place tasks preferentially in "high-affinity topological subsets." For example:

- Data Parallelism (DDP/FSDP): each step does all-reduce to synchronize gradients, with large communication volume but amenable to pipelining. Groups are usually arranged as a ring or tree so each hop goes over NVLink. Place within the same node first, and within the same rack when crossing nodes.
- Tensor Parallelism: splits a single matrix multiplication across multiple cards, requiring all-reduce on every forward/backward step. Latency-sensitive, **almost mandatory to place within the same node over NVLink**.
- Pipeline Parallelism: passes activations between adjacent stages, a one-way flow. Communication volume is smaller than TP, but latency must be low — usually placed on adjacent GPUs.
- Expert Parallelism (MoE): each step routes tokens to remote experts, with an all-to-all communication pattern that is extremely sensitive to both bandwidth and topology; expanded separately below.


In [ ]:
# === Sensitivity of different parallel strategies to bandwidth ===
strategies = [
    # (strategy, per-step comm pattern, comm volume, topology requirement)
    ("Data Parallelism (DDP/FSDP)",   "all-reduce gradients",       "large",     "Intra-node first, use IB across nodes"),
    ("Tensor Parallelism",            "all-reduce activations",     "medium",    "Forced intra-node (NVLink)"),
    ("Pipeline Parallelism",          "P2P activations",            "small-med", "Adjacent GPUs suffice"),
    ("Expert Parallelism",            "all-to-all token routing",   "large",     "Intra-node first, cross-node needs full NVLink+IB"),
]

print(f"{'Strategy':<32} {'Comm pattern':<26} {'Volume':<10} Topology requirement")
print("-" * 100)
for s, mode, vol, req in strategies:
    print(f"{s:<32} {mode:<26} {vol:<10} {req}")
print()
print("Key observation: TP essentially cannot cross nodes; EP has the heaviest communication pattern and the highest topology requirement.")


## 6. Expert Parallelism in MoE: Why It Is So Sensitive to Network Topology

A Mixture of Experts (MoE) model replaces the FFN layer with several parallel expert networks, where each token activates only a few of them (typically 2: 1 shared + 1 routed). Expert Parallelism (EP) is a parallel strategy that distributes different experts across different GPUs. For example, 8 experts are placed across 8 cards, with each card responsible for the forward and backward of 1 expert.

The communication pattern of EP is **all-to-all**:

1. Each GPU computes which experts the local tokens should route to
2. Tokens whose target experts are on remote GPUs are sent over the network
3. Remote GPUs receive the tokens and run the expert forward pass
4. The results are sent back to the original GPU via another all-to-all

Each training step requires two round-trip all-to-all passes, and the data volume communicated equals batch_size x seq_len x hidden_dim x number of activated experts. This communication pattern is extremely sensitive to both bandwidth and topology, because every GPU may need to exchange tokens with any other GPU.

The cost of cross-node EP is especially high. Suppose 8 experts are distributed across two nodes, with 4 experts per node. Tokens routed locally go over NVLink (450 GB/s), while cross-node tokens go over IB (50 GB/s per single link). If tokens are uniformly distributed across experts, about 3/8 of the tokens must cross nodes — this portion becomes the bottleneck. This is why large-scale MoE training (e.g., DeepSeek-MoE, Mixtral) almost always requires fully provisioned IB, and the scheduler prioritizes placing GPUs of the same EP group within the same node or the same rack.


In [ ]:
# === EP all-to-all communication volume estimation ===
# Assumption: 8-GPU node, 1 expert per card, tokens in the batch route uniformly to 8 experts
# Each token is a hidden_dim-dimensional vector

batch_size = 4096        # Global batch
seq_len = 4096
hidden_dim = 4096        # Model hidden size
expert_topk = 2          # Each token activates 2 experts (including shared)
bf16_bytes = 2

# Each token is sent topk times (routed to topk experts)
# all-to-all one direction: each card sends batch*seq*topk/8 tokens, each token is hidden_dim-dimensional
tokens_per_card = batch_size * seq_len * expert_topk / 8
bytes_per_token = hidden_dim * bf16_bytes

# Bytes sent per card in one direction
send_bytes = tokens_per_card * bytes_per_token
# all-to-all one direction + one direction back = 2x
round_trip = send_bytes * 2

print(f"EP communication estimation (batch={batch_size}, seq={seq_len}, hidden={hidden_dim}, topk={expert_topk}):")
print(f"  Tokens sent per card:    {tokens_per_card:.0f}")
print(f"  Size per token:          {bytes_per_token} bytes")
print(f"  Per-card one-way send:   {send_bytes / 1e9:.2f} GB")
print(f"  Round-trip volume:       {round_trip / 1e9:.2f} GB")
print()
print(f"Transfer time (NVLink 4.0, 450 GB/s):   {round_trip / 450:.3f} s")
print(f"Transfer time (IB NDR single link, 50 GB/s): {round_trip / 50:.3f} s")
print()
print("Cross-node transfer time is about 9x that of intra-node.")
print("In MoE training, if EP crosses nodes, all-to-all is often the dominant time cost of a single training step.")


### EP Deployment Strategies

Because EP is extremely sensitive to bandwidth, several common deployment patterns exist in production:

- **Intra-node EP**: limit the number of experts to within a node (8 experts on 8 cards, or 16 experts). All all-to-all goes over NVLink. The advantage is speed; the disadvantage is that model size is constrained.
- **Cross-node EP + NVLink-first scheduling**: cross-node EP must saturate IB, and the scheduler places GPUs of the same EP group on the "network-closest" GPUs. NCCL automatically inspects the `nvidia-smi topo -m` output and preferentially selects NVLink paths.
- **Expert Duplication**: replicate hot experts onto multiple GPUs to reduce cross-node routing. The cost is parameter redundancy.
- **Dedicated communication libraries like DeepEP**: DeepSeek's open-source MoE communication library, which does low-latency optimization for EP's all-to-all and supports hybrid scheduling of intra-node NVLink + cross-node IB.


## 7. In Practice: Commands to Inspect GPU Topology

On a real machine, several NVIDIA tools let you directly see the hardware information described above. Below are the most commonly used commands; the example outputs come from a typical 8-GPU H100 node.


### nvidia-smi: Basic Information

`nvidia-smi` is a command bundled with the NVIDIA driver, showing the GPU's model, memory, temperature, utilization, and so on. Running it without arguments shows an overview of all GPUs:

```bash
nvidia-smi
```

For continuous monitoring (similar to `top`), use `nvtop` or `watch -n 1 nvidia-smi`.


### nvidia-smi topo -m: View the Inter-GPU Interconnect Topology

This is the most important command when troubleshooting multi-GPU training issues. It displays an 8x8 matrix where each row and column represents a GPU, and the symbol in each cell indicates how those two GPUs are connected:


In [ ]:
# === Example output of nvidia-smi topo -m (8-GPU H100 node) ===
# On a real machine run: nvidia-smi topo -m
# Below is a string simulation of typical output

topo_output = '''
        GPU0    GPU1    GPU2    GPU3    GPU4    GPU5    GPU6    GPU7
GPU0     X      NV12    NV12    NV12    NV12    NV12    NV12    NV12
GPU1    NV12     X      NV12    NV12    NV12    NV12    NV12    NV12
GPU2    NV12    NV12     X      NV12    NV12    NV12    NV12    NV12
GPU3    NV12    NV12    NV12     X      NV12    NV12    NV12    NV12
GPU4    NV12    NV12    NV12    NV12     X      NV12    NV12    NV12
GPU5    NV12    NV12    NV12    NV12    NV12     X      NV12    NV12
GPU6    NV12    NV12    NV12    NV12    NV12    NV12     X      NV12
GPU7    NV12    NV12    NV12    NV12    NV12    NV12    NV12     X

Legend:

  X    = Self
  SYS  = Connection traversing PCIe as well as the SMP interconnect between NUMA nodes
  NODE = Connection traversing PCIe as well as the interconnect between PCIe Host Bridges within a NUMA node
  PHB  = Connection traversing PCIe as well as a PCI Host Bridge (typically the CPU)
  PXB  = Connection traversing multiple PCI switches (PCIe bridge)
  PIX  = Connection traversing a single PCI switch (PCIe bridge)
  NV#  = Connection traversing a bonded set of # NVLinks
'''

print(topo_output)
print("Symbol interpretation:")
print("  NV12 means the two cards have 12 NVLink channels between them (typical H100 configuration)")
print("  PIX/PXB means a connection within the same PCIe switch, with lower bandwidth")
print("  SYS means crossing NUMA nodes, the slowest communication")
print()
print("If the matrix shows lots of PIX or SYS, NVLink is not enabled and multi-GPU training will be much slower.")


### nccl-tests: Measuring Actual Communication Bandwidth

NVIDIA's NCCL (NVIDIA Collective Communications Library) provides the `nccl-tests` toolset, which can measure the actual bandwidth of collective operations like all-reduce, all-gather, and all-to-all. The command below tests 8-GPU all-reduce:

```bash
# After building nccl-tests, run all_reduce_perf
./build/all_reduce_perf -b 8 -e 1G -f 2 -g 8
```

The output shows algbw (algorithm bandwidth) and busbw (bus bandwidth) for different message sizes. busbw is the key metric for evaluating multi-GPU interconnect efficiency — generally, the all-reduce busbw on an H100 node should approach 80% or more of the NVLink bandwidth.


In [ ]:
# === Example output of a typical all_reduce_perf run ===
nccl_output = """
# size    count   type  redOp   time  algbw  busbw  error
        (B)    (elements)            (us)  (GB/s) (GB/s)
        8          2  float    sum   xx.x   x.xx   x.xx  0e+00
       16          4  float    sum   xx.x   x.xx   x.xx  0e+00
...
  8388608    2097152  float    sum   xx.x  xxx.x  xxx.x  0e+00
 16777216    4194304  float    sum   xx.x  xxx.x  xxx.x  0e+00
...
Out of bounds values : 0 OK
Avg bus bandwidth    : xxx.xx GB/s
"""

print(nccl_output)
print("Health indicators:")
print("  - all-reduce busbw on 8 H100s with full NVLink interconnect is typically 200-300 GB/s")
print("  - all-reduce busbw across nodes over IB NDR is typically 30-50 GB/s")
print("  - If measured bandwidth is far below spec, check NVLink status, NCCL version, and whether P2P is enabled")


## 8. Tying It Together: How Hardware Characteristics Drive Algorithm Choices

Connecting this section's content back to the earlier distributed-training discussion, here are the key correspondences:

- **Flash Attention is necessary** because SRAM is nearly 10x faster than HBM; tiling attention to stay in SRAM is the key to performance.
- **Tensor Parallelism must stay intra-node** because every step does all-reduce, and NVLink is 6-10x faster than IB.
- **Pipeline Parallelism can cross nodes** because adjacent-stage communication is small, and IB bandwidth suffices.
- **FSDP/ZeRO can cross nodes** because pipelined all-reduce can fully utilize aggregate IB bandwidth, but busbw is the key metric.
- **MoE EP must use fully provisioned IB** because all-to-all is the heaviest communication pattern and cross-node becomes the main bottleneck.
- **FP8 training is popular on Hopper** because FP8 Tensor Core throughput is 2x BF16, with memory footprint and bandwidth halved.


In [ ]:
# === Hardware characteristic -> algorithm choice correspondence table ===
mapping = [
    ("Flash Attention",          "SRAM is 10x faster than HBM",        "Tile attention to stay in SRAM"),
    ("Tensor Parallelism",       "NVLink is 6-10x faster than IB",     "TP group must be intra-node"),
    ("Pipeline Parallelism",     "P2P communication is small",         "Can cross nodes, adjacent stages suffice"),
    ("FSDP / ZeRO-3",            "all-reduce is pipelined",            "Cross-node viable, busbw is key"),
    ("MoE Expert Parallelism",   "all-to-all is the heaviest pattern", "Must fully provision IB, intra-node first"),
    ("FP8 training",             "Hopper FP8 Tensor Core",             "2x BF16 throughput, memory and bandwidth halved"),
]

print(f"{'Algorithm / Strategy':<30} {'Hardware basis':<36} Choice")
print("-" * 95)
for algo, hw, choice in mapping:
    print(f"{algo:<30} {hw:<36} {choice}")


## Summary

Confirm you understand the following:

- [ ] HBM has large capacity but is relatively slow; SRAM has small capacity but is extremely fast (on H100, SRAM estimated bandwidth is about 10x that of HBM)
- [ ] Flash Attention tiles attention to stay in SRAM, reducing HBM read/writes from $O(N^2)$ to $O(N)$
- [ ] A GPU is composed of several SMs, each containing Tensor Cores; a warp is the 32-thread scheduling unit
- [ ] Hopper introduced FP8 (E4M3, E5M2) Tensor Cores, doubling throughput versus BF16 and halving memory bandwidth
- [ ] PCIe 5.0 is 64 GB/s unidirectional, NVLink 4.0 is 450 GB/s bidirectional — a gap of about 6-10x
- [ ] NVSwitch enables intra-node 8-GPU full interconnect; cross-node uses InfiniBand (NDR 400 Gbps)
- [ ] DGX H100 intra-node full-interconnect bandwidth is about 28 x 450 GB/s; cross-node aggregate is about 8 x 50 GB/s
- [ ] Tensor Parallelism essentially must be intra-node; EP all-to-all is the most topology-sensitive
- [ ] `nvidia-smi topo -m` is the first command to reach for when troubleshooting multi-GPU topology issues


## Exercises

> You may ask AI to help explain ideas and check directions, but it is not recommended to have AI "solve the whole exercise for you."

**Exercise 1: Estimate the HBM Read/Write Savings of Flash Attention**

Sequence length $N = 8192$, hidden_dim $d = 4096$, number of heads $h = 32$. The intermediate matrix $QK^T$ of naive attention has shape $[h, N, N]$ stored in BF16. Compute the number of bytes of this intermediate matrix, and explain how much Flash Attention can reduce it to.

Hint: the naive implementation writes $[h, N, N]$ to HBM and reads it back; Flash Attention uses online softmax and tiled computation, keeping only the $[h, N]$ normalization factors and accumulation results.


In [ ]:
# Exercise 1: HBM read/write savings of Flash Attention
N = 8192
d = 4096
h = 32
bf16_bytes = 2

# TODO: compute the byte count of the naive attention intermediate matrix (in GB)
naive_bytes_gb = None

# TODO: byte count of the intermediate result retained by Flash Attention (in GB)
# Hint: online softmax only keeps [h, N] running max and running sum
flash_bytes_gb = None

assert naive_bytes_gb is not None, "Please compute the naive attention intermediate matrix size first"
assert flash_bytes_gb is not None, "Please compute the Flash Attention intermediate result size first"

expected_naive = h * N * N * bf16_bytes / 1e9
expected_flash = h * N * 2 * 2 * bf16_bytes / 1e9   # running max + running sum, each h*N elements

assert abs(naive_bytes_gb - expected_naive) < 0.1, f"Naive intermediate matrix should be {expected_naive:.2f} GB"
assert abs(flash_bytes_gb - expected_flash) < 0.001, f"Flash intermediate result should be {expected_flash:.4f} GB"

print(f"✅ Exercise 1 passed:")
print(f"   Naive attention intermediate matrix: {naive_bytes_gb:.2f} GB")
print(f"   Flash Attention intermediate result: {flash_bytes_gb:.4f} GB")
print(f"   Savings ratio: {1 - flash_bytes_gb / naive_bytes_gb * 100:.2f}%")


**Exercise 2: Cross-node vs Intra-node Transfer Time**

The BF16 parameters of a 70B model are to be transferred between two GPUs. If the two cards are in the same node (over NVLink 4.0), what is the transfer time in seconds? If they are across nodes (over a single IB NDR link), what is the transfer time? How many times faster is one over the other?

Hint: NVLink 4.0 bidirectional bandwidth is 450 GB/s, and a single IB NDR link is 50 GB/s. The parameter count is 70e9, and BF16 is 2 bytes per param.


In [ ]:
# Exercise 2: cross-node vs intra-node transfer time
P = 70e9
bf16_bytes = 2
param_bytes = P * bf16_bytes

nvlink_bw = 450       # NVLink 4.0 bidirectional GB/s
ib_ndr_bw = 50        # IB NDR single link GB/s

# TODO: compute the two transfer times (in seconds)
t_intra = None       # Intra-node NVLink
t_inter = None       # Cross-node IB

assert t_intra is not None and t_inter is not None, "Please compute both transfer times first"

expected_intra = param_bytes / (nvlink_bw * 1e9)
expected_inter = param_bytes / (ib_ndr_bw * 1e9)
assert abs(t_intra - expected_intra) < 0.01, f"Intra-node time should be {expected_intra:.2f} s"
assert abs(t_inter - expected_inter) < 0.01, f"Cross-node time should be {expected_inter:.2f} s"

print(f"✅ Exercise 2 passed:")
print(f"   Total params: {param_bytes / 1e9:.0f} GB")
print(f"   Intra-node (NVLink 4.0): {t_intra:.2f} s")
print(f"   Cross-node (IB NDR):     {t_inter:.2f} s")
print(f"   Speed gap: {t_inter / t_intra:.1f}x")


**Exercise 3: Diagnosing a Topology Problem**

Suppose you run `nvidia-smi topo -m` on an 8-GPU node and the output matrix is full of `PIX` instead of `NV12`. What would you observe when running DDP training? What should you check first?

Hint: `PIX` means two GPUs are connected through the same PCIe switch without going over NVLink. NVLink not working is usually related to drivers, CUDA version, power mode, or PCIe topology configuration.


In [ ]:
# Exercise 3: diagnosing a topology problem (conceptual, fill-in)
# Question: when topo -m shows all PIX, what happens to DDP training?

# Fill in: put the correct strings into the two variables below
# Phenomenon (choose from options_phenomenon):
#   "Training speed is normal, consistent with NVLink"
#   "Training speed drops sharply, all-reduce becomes the bottleneck"
#   "Training cannot start, NCCL reports an error"
phenomenon = None

# What to check first (choose from options_check):
#   "Whether the NVIDIA driver supports NVLink, whether the CUDA version matches, whether the NVLink bridge is enabled in BIOS"
#   "Increase batch size"
#   "Change the PyTorch version"
check_action = None

options_phenomenon = {
    "Training speed drops sharply, all-reduce becomes the bottleneck"
}
options_check = {
    "Whether the NVIDIA driver supports NVLink, whether the CUDA version matches, whether the NVLink bridge is enabled in BIOS"
}

assert phenomenon in options_phenomenon, "Phenomenon is wrong: PIX means NVLink is not enabled, all-reduce can only go over much slower PCIe"
assert check_action in options_check, "Wrong direction to investigate: first confirm whether NVLink is healthy at the hardware and driver level"

print("✅ Exercise 3 passed:")
print(f"   Phenomenon: {phenomenon}")
print(f"   Investigate: {check_action}")


## References

- NVIDIA, [H100 Tensor Core GPU Architecture Whitepaper](https://resources.nvidia.com/en-us-tensor-core), 2022
- NVIDIA, [Hopper FP8 Tensor Cores](https://www.nvidia.com/en-us/data-center/hopper-architecture/), 2022
- NVIDIA, [NVLink and NVSwitch](https://www.nvidia.com/en-us/data-center/nvlink/), 2023
- NVIDIA, [DGX H100 System Architecture](https://www.nvidia.com/en-us/data-center/dgx-h100/), 2023
- Dao et al., [FlashAttention: Fast and Memory-Efficient Exact Attention with IO-Awareness](https://arxiv.org/abs/2205.14135), 2022
- NVIDIA, [NCCL Tests Documentation](https://github.com/NVIDIA/nccl-tests), 2023
- InfiniBand Trade Association, [InfiniBand Architecture Specification](https://www.infinibandta.org/), 2024
